In [11]:
import os
import sys
import google.generativeai as genai
# REMOVIDO: import pandas as pd
# REMOVIDO: import io
# REMOVIDO: import json
# REMOVIDO: import re (pode ser adicionado se precisar limpar o markdown bruto)

# --- 1. Configuração Segura da Chave da API ---
# (Código mantido igual)
print("--- Configurando API Key ---")
api_key = os.environ.get('API_KEY_GEMINI')
if not api_key:
    print("\n!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!")
    print("!!! ERRO CRÍTICO: Variável de ambiente 'API_KEY_GEMINI' não definida.")
    # ... (restante da mensagem) ...
    print("!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!\n")
    sys.exit(1)

try:
    genai.configure(api_key=api_key)
    print("Chave da API configurada com sucesso.")
except Exception as e:
    print(f"Erro ao configurar a API do Google Generative AI: {e}")
    sys.exit(1)


--- Configurando API Key ---
Chave da API configurada com sucesso.


In [13]:
# --- 2. Configurações Gerais ---
print("\n--- Configurações do Script ---")
INPUT_FOLDER = "Downloads/Documentos Recortados/ppc_fed_md" # <--- DEFINA SUA PASTA AQUI
PROCESS_MODE = "all"  # Opções: "all", "count", "specific"
MAX_FILES_TO_PROCESS = 5 # Usado apenas se PROCESS_MODE = "count"
SPECIFIC_FILES = ["2-ppc_lic_unifap.md"] # Usado apenas se PROCESS_MODE = "specific"

# --- ALTERADO: Nome do arquivo de saída Markdown ---
OUTPUT_MARKDOWN_FILE = "resultados_brutos_2.5pro_teste.md" # <--- Nome do arquivo Markdown para salvar os resultados brutos
# ----------------------------------------------------

# --- Configurações do Modelo Gemini ---
MODEL_NAME = "gemini-2.5-pro"
MAX_OUTPUT_TOKENS = 4096
TEMPERATURE = 0.4 # Ajuste conforme necessário para a qualidade do Markdown
TOP_K = 40
TOP_P = 0.90

print(f"Pasta de Entrada: {INPUT_FOLDER}")
print(f"Modo de Processamento: {PROCESS_MODE}")
# ... (impressões de config count/specific) ...
print(f"Modelo Gemini: {MODEL_NAME}")
print(f"Salvar Saída Bruta em: {OUTPUT_MARKDOWN_FILE}")
# ----------------------------------------------


--- Configurações do Script ---
Pasta de Entrada: Downloads/Documentos Recortados/ppc_fed_md
Modo de Processamento: all
Modelo Gemini: gemini-2.5-pro
Salvar Saída Bruta em: resultados_brutos_2.5pro_teste.md


In [15]:
# --- 3. Definição da Instrução do Sistema e Pergunta ---
# Instrução do sistema (mantida)
instruction =  (
    "Como especialista em análise de documentos acadêmicos (PPCs - Projetos Pedagógicos de Curso) e em Educação Física Adaptada, sua tarefa é analisar o texto fornecido e extrair **APENAS** as informações **explicitamente solicitadas** na pergunta. "
    "Não adicione interpretações, comentários, introduções ou conclusões. Seja conciso e direto. "
    "Use as seguintes definições como guia para identificar as disciplinas relevantes: "
    "(1) Educação Física Adaptada: Foco em adaptar conteúdos de EF para alunos com deficiências/transtornos. "
    "(2) Atividade Física Adaptada: Foco em modificar exercícios/atividades para pessoas com deficiências/transtornos. "
    "(3) Esporte Adaptado: Foco em modalidades esportivas modificadas/criadas para pessoas com deficiência. "
    "Ignore disciplinas focadas primariamente em saúde pública geral, doenças crônicas (sem o vínculo claro com adaptação para deficiência), ou inclusão social/étnico-racial (sem o vínculo claro com adaptação para deficiência)."
)


# Pergunta específica (com exemplo Markdown de volta)
question =  (
 "Do texto do PPC fornecido, extraia **apenas** as disciplinas que se encaixam estritamente nas definições de Educação Física Adaptada, Atividade Física Adaptada ou Esporte Adaptado, conforme definido na instrução do sistema. "
    "Para cada disciplina encontrada, informe: "
    "1. Nome da Disciplina. "
    "2. Carga Horária Teórica. "
    "3. Carga Horária Prática. "
    "4. Carga Horária de Extensão (se aplicável). "
    "5. Carga Horária Total. "
    "6. Descrição (Ementa). "
    "7. Referências Bibliográficas Básicas. "
    "8. Referências Bibliográficas Complementares. "
    "Organize a resposta final **exclusivamente** em formato de tabela Markdown. "
    "Se alguma informação específica (ex: carga horária de extensão, bibliografia complementar) não for encontrada no texto para uma disciplina listada, preencha a célula correspondente com 'NA'. "
    
)


# --- 4. Instanciação do Modelo Generativo ---
print("\n--- Instanciando Modelo Generativo ---")
try:
    # Apenas um modelo necessário agora
    model = genai.GenerativeModel(
        model_name=MODEL_NAME,
        system_instruction=instruction,
        safety_settings={ # Ajuste conforme necessidade
            "HARM_CATEGORY_HARASSMENT": "BLOCK_NONE",
            "HARM_CATEGORY_HATE_SPEECH": "BLOCK_NONE",
        }
    )
    print(f"Modelo '{MODEL_NAME}' instanciado com sucesso.")
except Exception as e:
    print(f"Erro ao instanciar o modelo Generative AI: {e}")
    sys.exit(1)


--- Instanciando Modelo Generativo ---
Modelo 'gemini-2.5-pro' instanciado com sucesso.


In [17]:

# --- 5. Descoberta e Seleção de Arquivos ---
# (Código mantido igual)
print("\n--- Localizando Arquivos Markdown ---")
files_to_process = [] # Populado pela lógica abaixo
if not os.path.isdir(INPUT_FOLDER):
     print(f"ERRO: A pasta de entrada '{INPUT_FOLDER}' não existe.")
     sys.exit(1)
try:
    all_files_in_folder = [f for f in os.listdir(INPUT_FOLDER) if os.path.isfile(os.path.join(INPUT_FOLDER, f))]
    markdown_files_found = sorted([f for f in all_files_in_folder if f.lower().endswith(".md")])

    if not markdown_files_found:
        print(f"AVISO: Nenhum arquivo .md encontrado em '{INPUT_FOLDER}'.")
    elif PROCESS_MODE == "all":
        files_to_process = markdown_files_found
        print(f"Encontrados {len(files_to_process)} arquivos .md para processar.")
    # ... (lógica de seleção 'count' e 'specific') ...
    elif PROCESS_MODE == "count":
        count = min(MAX_FILES_TO_PROCESS, len(markdown_files_found))
        files_to_process = markdown_files_found[:count]
        print(f"Selecionados os primeiros {len(files_to_process)} de {len(markdown_files_found)} arquivos .md encontrados.")
    elif PROCESS_MODE == "specific":
        # ... (lógica specific mantida) ...
        files_to_process = [f for f in markdown_files_found if f in SPECIFIC_FILES]
        found_specific = len(files_to_process)
        requested_specific = len(SPECIFIC_FILES)
        print(f"Encontrados {found_specific} dos {requested_specific} arquivos .md especificados.")
        if found_specific < requested_specific:
            missing = [f for f in SPECIFIC_FILES if f not in files_to_process]
            print(f"  Arquivos especificados não encontrados ou não são .md: {missing}")
    else:
         print(f"ERRO: Modo de processamento '{PROCESS_MODE}' inválido.")
         sys.exit(1)

except OSError as e:
    print(f"Erro ao acessar a pasta de entrada '{INPUT_FOLDER}': {e}")
    sys.exit(1)


# --- 6. Loop Principal de Processamento ---
print("\n--- Iniciando Processamento dos Arquivos ---")
all_markdown_results = [] # Lista para armazenar os resultados brutos de Markdown

if not files_to_process:
    print("Nenhum arquivo para processar. Encerrando.")
else:
    for filename in files_to_process:
        filepath = os.path.join(INPUT_FOLDER, filename)
        print(f"\n--- Processando Arquivo: {filename} ---")

        # --- 6.1 Ler Conteúdo do Arquivo ---
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                file_content = f.read()
            print(f"Arquivo '{filename}' lido com sucesso.")
            if not file_content.strip():
                 print("AVISO: Arquivo está vazio. Pulando.")
                 continue
        except Exception as e: # Simplificado
             print(f"ERRO ao ler o arquivo '{filename}': {e}. Pulando.")
             continue

        # --- 6.2 Construir Prompt e Chamar API ---
        # Usa a 'question' que pede explicitamente Markdown
        prompt = f"**Contexto (Texto extraído do PPC - {filename}):**\n```markdown\n{file_content}\n```\n\n**Tarefa (Pergunta):**\n{question}\n\n**Resposta (estritamente no formato solicitado):**"

        print("Enviando requisição para a API Gemini...")
        raw_markdown_output = None
        try:
            response = model.generate_content(
                prompt,
                generation_config=genai.types.GenerationConfig(
                    candidate_count=1,
                    max_output_tokens=MAX_OUTPUT_TOKENS,
                    temperature=TEMPERATURE,
                    top_k=TOP_K,
                    top_p=TOP_P
                )
            )

            # Verificar bloqueio inicial do prompt
            try:
                if response.prompt_feedback.block_reason:
                    print(f"*** ALERTA: PROMPT BLOQUEADO para '{filename}'. Razão: {response.prompt_feedback.block_reason} ***")
                    # Adiciona uma entrada indicando o bloqueio ao resultado
                    all_markdown_results.append(f"## Resultados para: {filename}\n\n***PROMPT BLOQUEADO: {response.prompt_feedback.block_reason}***\n")
                    continue
            except Exception: pass

            # Tentar obter o texto bruto (que deve ser Markdown ou a frase 'Nenhuma...')
            raw_markdown_output = response.text.strip()
            print(f"Resposta bruta (Markdown esperado) recebida para '{filename}'.")

             # Armazenar o resultado bruto precedido pelo nome do arquivo
            result_entry = f"## Resultados para: {filename}\n\n{raw_markdown_output}\n"
            all_markdown_results.append(result_entry)

            # Opcional: Imprimir o resultado bruto no console para feedback imediato
            # print("\n--- Resultado Bruto Recebido ---")
            # print(raw_markdown_output)
            # print("--- Fim Resultado Bruto ---")


        except ValueError as ve:
            print(f"*** ERRO: CONTEÚDO DA RESPOSTA BLOQUEADO para '{filename}' (ValueError: {ve}). ***")
            reason = "Conteúdo da Resposta Bloqueado (ValueError)"
            try:
                reason += f" - Finish Reason: {response.candidates[0].finish_reason}"
            except Exception: pass
            all_markdown_results.append(f"## Resultados para: {filename}\n\n***ERRO: {reason}***\n")
            continue
        except AttributeError:
             print(f"*** ERRO: Resposta da API para '{filename}' não contém texto (AttributeError). Provavelmente bloqueada ou mal formada. ***")
             reason = "Resposta sem texto (AttributeError)"
             try:
                 reason += f" - Finish Reason: {response.candidates[0].finish_reason}"
             except Exception: pass
             all_markdown_results.append(f"## Resultados para: {filename}\n\n***ERRO: {reason}***\n")
             continue
        except Exception as e:
             print(f"\nErro CRÍTICO durante a chamada 'generate_content' para o arquivo '{filename}': {e}")
             all_markdown_results.append(f"## Resultados para: {filename}\n\n***ERRO CRÍTICO NA API: {e}***\n")
             continue

        # --- REMOVIDO: Etapa de Formatação e Criação de DataFrame ---

# --- 7. Consolidação e Saída dos Resultados (em Markdown) ---
print("\n--- Consolidação Final dos Resultados Brutos ---")

if not all_markdown_results:
    print("Nenhuma resposta foi coletada de nenhum arquivo.")
else:
    print(f"Consolidando {len(all_markdown_results)} respostas brutas...")
    # Junta todos os resultados com um separador claro entre eles
    final_markdown_content = "\n---\n".join(all_markdown_results)

    print(f"Salvando resultados brutos em '{OUTPUT_MARKDOWN_FILE}'...")
    try:
        with open(OUTPUT_MARKDOWN_FILE, 'w', encoding='utf-8') as f:
            f.write(f"# Análise Bruta de PPCs - {MODEL_NAME}\n\n") # Adiciona um título ao arquivo
            f.write(final_markdown_content)
        print(f"Resultados brutos salvos com sucesso em '{OUTPUT_MARKDOWN_FILE}'.")
        print("Você pode abrir este arquivo com um editor de texto ou visualizador de Markdown.")

    except IOError as e:
        print(f"Erro de I/O ao salvar o arquivo Markdown '{OUTPUT_MARKDOWN_FILE}': {e}")
    except Exception as e:
        print(f"Erro inesperado ao salvar o arquivo Markdown '{OUTPUT_MARKDOWN_FILE}': {e}")

print("\n--- Fim da Execução do Script ---")


--- Localizando Arquivos Markdown ---
Encontrados 112 arquivos .md para processar.

--- Iniciando Processamento dos Arquivos ---

--- Processando Arquivo: 1-ppc_lic_unir.md ---
Arquivo '1-ppc_lic_unir.md' lido com sucesso.
Enviando requisição para a API Gemini...
*** ERRO: Resposta da API para '1-ppc_lic_unir.md' não contém texto (AttributeError). Provavelmente bloqueada ou mal formada. ***

--- Processando Arquivo: 10-ppc_lic_if_toc_palmas.md ---
Arquivo '10-ppc_lic_if_toc_palmas.md' lido com sucesso.
Enviando requisição para a API Gemini...
Resposta bruta (Markdown esperado) recebida para '10-ppc_lic_if_toc_palmas.md'.

--- Processando Arquivo: 100-ppc_lic_ufvjm.md ---
Arquivo '100-ppc_lic_ufvjm.md' lido com sucesso.
Enviando requisição para a API Gemini...
Resposta bruta (Markdown esperado) recebida para '100-ppc_lic_ufvjm.md'.

--- Processando Arquivo: 101-ppc_bac_furg.md ---
Arquivo '101-ppc_bac_furg.md' lido com sucesso.
Enviando requisição para a API Gemini...
Resposta bruta (M